Aim: extract geospatial data from Open Street Map (OSM) files in downloaded from Geofabrik website in PBF format. Currently sertup for working for lines and filtering by specific tags. Options to have all tags were problematic as part of the chain uses geodataframes where duplicate tags in different cases were converted to lower case causing errors from duplicate headings.

In [1]:
import osmium
import geopandas as gpd
from shapely.geometry import LineString
import os
import time

In [2]:
# Select what to extract: 'highways' or 'railway'
extract_type = 'railway'  # change to 'highway' if needed


In [ ]:
import osmium
import geopandas as gpd
from shapely.geometry import LineString
import os
import time

class geometryExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.highways = []
        self.railways = []
        self.count = 0
        self.error_count = 0
        self.start_time = time.time()
        self.debug_samples = []

    def way(self, w):
        if extract_type in w.tags:
            self.count += 1
            if self.count % 10000 == 0:
                elapsed = time.time() - self.start_time
                print(f"Processed {self.count} {extract_type}s in {elapsed:.2f} seconds")
            if self.count <= 5:
                print(f"Way {w.id}: {len(w.nodes)} nodes, first node: {w.nodes[0].ref}")
            try:
                coords = [(n.lon, n.lat) for n in w.nodes if hasattr(n, 'lon') and hasattr(n, 'lat')]
                if len(coords) >= 2:
                    way_dict = {
                        'id': w.id,
                        extract_type: w.tags.get(extract_type, ''),
                        'name': w.tags.get('name', ''),
                        'geometry': LineString(coords)
                    }

                    if extract_type == 'highway':
                        way_dict.update({
                            'id': w.id,
                            'highway': w.tags.get('highway', ''),
                            'surface': w.tags.get('surface', ''),
                            'name': w.tags.get('name', ''),
                            'geometry': LineString(coords)
                        })
                        self.highways.append(way_dict)
                    elif extract_type == 'railway':
                        way_dict.update({
                            'tunnel': w.tags.get('tunnel', ''),
                            'bridge': w.tags.get('bridge', ''),
                            'electrified': w.tags.get('electrified', ''),
                            'usage': w.tags.get('usage', ''),
                            'service': w.tags.get('service', ''),
                            'layer': w.tags.get('layer', ''),
                        })
                        self.railways.append(way_dict)

                else:
                    if len(self.debug_samples) < 3:
                        self.debug_samples.append(w.id)
                    self.error_count += 1
            except Exception:
                self.error_count += 1



In [4]:
input_dir = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\raw\roads\osm\osm_regional_250521"
output_dir = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\railways\osm\osm_regional_250521"

os.makedirs(output_dir, exist_ok=True)

# Find input files that don't have corresponding output
input_files = [f for f in os.listdir(input_dir) if f.endswith("-latest.osm.pbf")]
files_to_process = []

for file in input_files:
    country_name = file.replace("-latest.osm.pbf", "").replace("-", "_")
    input_path = os.path.join(input_dir, file)
    if extract_type == "railway":
        output_path = os.path.join(output_dir, f"{country_name}_railways.gpkg")
    elif extract_type == "highway":
        output_path = os.path.join(output_dir, f"{country_name}_highways.gpkg")
    else:
        print("no extract type chosen")
    if os.path.exists(output_path):
        print(f"Output already exists for {country_name}. Skipping.")
    else:
        files_to_process.append((input_path, output_path))
        print(f"Will process: {country_name}")

print(f"Found {len(files_to_process)} files to process out of {len(input_files)}")

Will process: africa
Will process: antarctica
Will process: asia
Will process: australia_oceania
Will process: central_america
Will process: europe
Will process: north_america
Will process: south_america
Found 8 files to process out of 8


In [5]:
# Process each file
for input_path, output_path in files_to_process:
    file = os.path.basename(input_path)
    country_name = file.replace("-latest.osm.pbf", "").replace("-", "_")
    print(f"\nProcessing: {file} -> {output_path}")

    try:
        osm = osmium.io.Reader(input_path)
        idx = osmium.index.create_map("sparse_mem_array")
        lh = osmium.NodeLocationsForWays(idx)
        handler = geometryExtractor()
        osmium.apply(osm, lh, handler)

        if handler.highways:
            gdf = gpd.GeoDataFrame(handler.highways, crs="EPSG:4326")
            gdf.to_file(output_path, driver="GPKG")
            print(f"Saved to {output_path}")
        else:
            print("No valid highways to save.")
        print(f"Found {len(handler.highways)} highways, Errors: {handler.error_count}")

        if handler.railways:
            gdf = gpd.GeoDataFrame(handler.railways, crs="EPSG:4326")
            gdf.to_file(output_path, driver="GPKG")
            print(f"Saved to {output_path}")
        else:
            print("No valid railways to save.")
        print(f"Found {len(handler.railways)} railways, Errors: {handler.error_count}")
    except Exception as e:
        print(f"Error processing {file}: {e}")

# NB when run for highways it took 256 minutes to process the 50 national files in Europe - total of ~23GB. SLow but no memory errors.  



Processing: africa-latest.osm.pbf -> C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\railways\osm\osm_regional_250521\africa_railways.gpkg
Way 4064202: 111 nodes, first node: 25326744
Way 4247103: 6 nodes, first node: 288679287
Way 4251316: 4 nodes, first node: 25412605
Way 4251408: 13 nodes, first node: 1554952897
Way 4251729: 30 nodes, first node: 2647695309
Processed 10000 railways in 84.24 seconds
Processed 20000 railways in 95.81 seconds
Processed 30000 railways in 128.81 seconds
Processed 40000 railways in 191.65 seconds


: 